In [1]:
import os
from typing import List, Optional, Tuple

import numpy as np
import pandas as pd
from Bio import SeqIO

In [2]:
AA_SEQS_COL_E = "../resources/colombia_data/AA_sequences/E_DENV{}_aa_E.fasta"
AA_SEQS_EXP_E = "../resources/experimental_data/AA_sequences/E_DENV{}_aa_E.fasta"
AA_SEQS_EXP_E_MAP = "../resources/experimental_data/AA_sequences/E_DENV{}_aa_E_map.fasta"
DIST_EPITOPE_GRANTHAM = "../output/distances_epitope_grantham_DENV{sero}.csv"
DIST_EPITOPE_GRANTHAM_VAL = "../output/validation_distances_epitope_grantham_DENV{sero}.csv"
DIST_ALL_GRANTHAM_VAL = "../output/validation_distances_all_grantham_DENV{sero}.csv"
DIST_EPITOPE_HAMMING_VAL = "../output/validation_distances_epitope_hamming_DENV{sero}.csv"
DIST_ALL_HAMMING_VAL = "../output/validation_distances_all_hamming_DENV{sero}.csv"

### Fasta de secuencias de referencia que SOLO se encuentran en el mapa

In [ ]:
valid_strain_ids = set(mapa_ag['strainID'].astype(str).tolist())
print(f"Total de cepas de referencia válidas en mapa_ag: {len(valid_strain_ids)}")

serotypes = [1, 2, 3, 4]

print("\nFiltrando secuencias experimentales...")
for sero in serotypes:
    input_fasta = AA_SEQS_EXP_E.format(sero)
    output_fasta = AA_SEQS_EXP_E_MAP.format(sero)
    
    all_records = list(SeqIO.parse(input_fasta, "fasta"))
    total_seqs = len(all_records)
    
    filtered_records = [r for r in all_records if r.id in valid_strain_ids]
    kept_seqs = len(filtered_records)
    
    SeqIO.write(filtered_records, output_fasta, "fasta")
    
    print(f"  DENV{sero}: Conservadas {kept_seqs} / {total_seqs} secuencias.")
    print(f"          -> Guardado en: {output_fasta}")

## Distancias

### Distancia de Hamming

In [31]:
def hamming_distance(s1, s2):
    if len(s1) != len(s2):
        raise ValueError("Strings must be of the same length")
    
    return sum(el1 != el2 for el1, el2 in zip(s1, s2))

### Distancia de Grantham

In [32]:
def grantham_distance(s1, s2):
    grantham_matrix = {
      'A': {'A':0,'R':112,'N':111,'D':126,'C':195,'Q':91,'E':107,'G':60,'H':86,'I':94,'L':96,'K':106,'M':84,'F':113,'P':27,'S':99,'T':58,'W':148,'Y':112,'V':64},
      'R': {'A':112,'R':0,'N':86,'D':96,'C':180,'Q':43,'E':54,'G':125,'H':29,'I':97,'L':102,'K':26,'M':91,'F':97,'P':103,'S':110,'T':71,'W':101,'Y':77,'V':96},
      'N': {'A':111,'R':86,'N':0,'D':23,'C':139,'Q':46,'E':42,'G':80,'H':68,'I':149,'L':153,'K':94,'M':142,'F':158,'P':91,'S':46,'T':65,'W':174,'Y':143,'V':133},
      'D': {'A':126,'R':96,'N':23,'D':0,'C':154,'Q':61,'E':45,'G':94,'H':81,'I':168,'L':172,'K':101,'M':160,'F':177,'P':108,'S':65,'T':85,'W':181,'Y':160,'V':152},
      'C': {'A':195,'R':180,'N':139,'D':154,'C':0,'Q':154,'E':170,'G':159,'H':174,'I':198,'L':198,'K':202,'M':196,'F':205,'P':169,'S':112,'T':149,'W':215,'Y':194,'V':192},
      'Q': {'A':91,'R':43,'N':46,'D':61,'C':154,'Q':0,'E':29,'G':87,'H':24,'I':109,'L':113,'K':53,'M':101,'F':116,'P':76,'S':68,'T':42,'W':130,'Y':99,'V':96},
      'E': {'A':107,'R':54,'N':42,'D':45,'C':170,'Q':29,'E':0,'G':98,'H':40,'I':134,'L':138,'K':56,'M':126,'F':140,'P':93,'S':80,'T':65,'W':152,'Y':122,'V':121},
      'G': {'A':60,'R':125,'N':80,'D':94,'C':159,'Q':87,'E':98,'G':0,'H':98,'I':135,'L':138,'K':127,'M':127,'F':153,'P':42,'S':56,'T':59,'W':184,'Y':147,'V':109},
      'H': {'A':86,'R':29,'N':68,'D':81,'C':174,'Q':24,'E':40,'G':98,'H':0,'I':94,'L':99,'K':32,'M':87,'F':100,'P':77,'S':89,'T':47,'W':115,'Y':83,'V':84},
      'I': {'A':94,'R':97,'N':149,'D':168,'C':198,'Q':109,'E':134,'G':135,'H':94,'I':0,'L':5,'K':102,'M':10,'F':21,'P':95,'S':142,'T':89,'W':61,'Y':33,'V':29},
      'L': {'A':96,'R':102,'N':153,'D':172,'C':198,'Q':113,'E':138,'G':138,'H':99,'I':5,'L':0,'K':107,'M':15,'F':22,'P':98,'S':145,'T':92,'W':61,'Y':36,'V':32},
      'K': {'A':106,'R':26,'N':94,'D':101,'C':202,'Q':53,'E':56,'G':127,'H':32,'I':102,'L':107,'K':0,'M':95,'F':102,'P':103,'S':121,'T':78,'W':110,'Y':85,'V':97},
      'M': {'A':84,'R':91,'N':142,'D':160,'C':196,'Q':101,'E':126,'G':127,'H':87,'I':10,'L':15,'K':95,'M':0,'F':28,'P':87,'S':135,'T':81,'W':67,'Y':36,'V':21},
      'F': {'A':113,'R':97,'N':158,'D':177,'C':205,'Q':116,'E':140,'G':153,'H':100,'I':21,'L':22,'K':102,'M':28,'F':0,'P':114,'S':155,'T':103,'W':40,'Y':22,'V':50},
      'P': {'A':27,'R':103,'N':91,'D':108,'C':169,'Q':76,'E':93,'G':42,'H':77,'I':95,'L':98,'K':103,'M':87,'F':114,'P':0,'S':74,'T':38,'W':147,'Y':110,'V':68},
      'S': {'A':99,'R':110,'N':46,'D':65,'C':112,'Q':68,'E':80,'G':56,'H':89,'I':142,'L':145,'K':121,'M':135,'F':155,'P':74,'S':0,'T':58,'W':177,'Y':144,'V':124},
      'T': {'A':58,'R':71,'N':65,'D':85,'C':149,'Q':42,'E':65,'G':59,'H':47,'I':89,'L':92,'K':78,'M':81,'F':103,'P':38,'S':58,'T':0,'W':128,'Y':92,'V':69},
      'W': {'A':148,'R':101,'N':174,'D':181,'C':215,'Q':130,'E':152,'G':184,'H':115,'I':61,'L':61,'K':110,'M':67,'F':40,'P':147,'S':177,'T':128,'W':0,'Y':37,'V':88},
      'Y': {'A':112,'R':77,'N':143,'D':160,'C':194,'Q':99,'E':122,'G':147,'H':83,'I':33,'L':36,'K':85,'M':36,'F':22,'P':110,'S':144,'T':92,'W':37,'Y':0,'V':55},
      'V': {'A':64,'R':96,'N':133,'D':152,'C':192,'Q':96,'E':121,'G':109,'H':84,'I':29,'L':32,'K':97,'M':21,'F':50,'P':68,'S':124,'T':69,'W':88,'Y':55,'V':0}
   }
    
    if len(s1) != len(s2):
        raise ValueError("Sequences must be of the same length")

    distance = 0

    for aa1, aa2 in zip(s1, s2):
        if aa1 == "-" or aa2 == "-":
            continue
        if aa1 == "*" or aa2 == "*":
            continue
        if aa1 not in grantham_matrix or aa2 not in grantham_matrix[aa1]:
            continue
        distance += grantham_matrix[aa1][aa2]
    return distance

### Distancia antigénica p-all-epitope

In [33]:
def antigenic_distance(s1, s2, sero, type='grantham'):

    # Posiciones en numeración 1-based (convención de la literatura)
    locations_EDI = [
        [47, 47], [123, 126], [148, 165], [167, 167],
        [170, 177], [180, 180], [279, 279], [293, 293]
    ]
    locations_EDII = [
        [67, 74], [76, 79], [88, 88], [97, 118],
        [215, 215], [219, 219], [233, 233], [237, 237],
        [244, 249], [256, 256], [266, 268], [272, 276]
    ]
    locations_EDIII = [
        [282, 282], [298, 311], [314, 370],
        [372, 372], [382, 385], [387, 390]
    ]

    # DENV3 E protein is 2 residues shorter due to a deletion at
    # positions 158-159 (relative to DENV1/2/4 numbering).
    # Every position > 159 is shifted by -2 (still 1-based).
    sero_id = str(sero).upper()
    if sero == 3 or '3' in sero_id:
        def _shift(loc):
            s, e = loc
            if s > 159:
                return [s - 2, e - 2]
            elif e > 159:
                # Region spans the deletion boundary
                return [s, e - 2]
            return [s, e]

        locations_EDI   = [_shift(l) for l in locations_EDI]
        locations_EDII  = [_shift(l) for l in locations_EDII]
        locations_EDIII = [_shift(l) for l in locations_EDIII]

    def _domain_distance(locations):
        total_dist = 0
        n_residues = 0
        for start, end in locations:
            # Conversión de 1-based inclusivo a slicing 0-based: [start-1 : end]
            if type == 'grantham':
                total_dist += grantham_distance(s1[start - 1:end],s2[start - 1:end])
            else:
                total_dist += hamming_distance(s1[start - 1:end],s2[start - 1:end])
            n_residues += end - start + 1
        return total_dist, n_residues

    sum_dist_EDI,   n_EDI   = _domain_distance(locations_EDI)
    sum_dist_EDII,  n_EDII  = _domain_distance(locations_EDII)
    sum_dist_EDIII, n_EDIII = _domain_distance(locations_EDIII)

    w = 215 if type == 'grantham' else 1

    # Ec. 2: E_i = (sum G_k) / (n_i * 215) * 20, por dominio
    E_EDI   = (sum_dist_EDI   / (n_EDI   * w)) * 20
    E_EDII  = (sum_dist_EDII  / (n_EDII  * w)) * 20
    E_EDIII = (sum_dist_EDIII / (n_EDIII * w)) * 20

    # Ec. 3: promedio de los tres dominios
    epit_dist = (E_EDI + E_EDII + E_EDIII) / 3
    return epit_dist

### Validar Corte de regiones epitópicas

In [34]:
def get(sero):
    return str(next(iter(SeqIO.parse(AA_SEQS_EXP_E_MAP.format(sero), "fasta"))).seq)

def chk(sero, sites):
    s = get(sero)
    off = lambda p: p - 2 if (sero == 3 and p > 159) else p   # deleción DENV3
    for p, aa in sites.items():
        q = off(p)
        print(f"DENV{sero} {aa}{p}: 1-based={s[q-1]} | 0-based={s[q]}  "
              f"{'OK' if s[q-1]==aa else '--'}")

# DENV2 (filas DV2-44, DV2-76, 3H2/2C8, 1A1D-2)
chk(2, {88:'K', 233:'Q', 244:'H', 282:'V', 309:'V', 310:'K',
        361:'K', 383:'E', 384:'P'})

# J8/J9, DENV1-4: K47, H149, V151, F279
for sero in [1, 2, 3, 4]:
    chk(sero, {47:'K', 149:'H', 151:'V', 279:'F'})

# DENV4, 1G6: 387-ALTLH-390 con T388 y H390
s4 = get(4)
print("DENV4 s[385:390] =", s4[385:390], "(esperado ALTLH, posiciones 386-390)")

DENV2 K88: 1-based=K | 0-based=R  OK
DENV2 Q233: 1-based=Q | 0-based=K  OK
DENV2 H244: 1-based=H | 0-based=A  OK
DENV2 V282: 1-based=H | 0-based=L  --
DENV2 V309: 1-based=V | 0-based=K  OK
DENV2 K310: 1-based=K | 0-based=E  OK
DENV2 K361: 1-based=K | 0-based=D  OK
DENV2 E383: 1-based=E | 0-based=P  OK
DENV2 P384: 1-based=P | 0-based=G  OK
DENV1 K47: 1-based=K | 0-based=T  OK
DENV1 H149: 1-based=H | 0-based=Q  OK
DENV1 V151: 1-based=V | 0-based=G  OK
DENV1 F279: 1-based=F | 0-based=A  OK
DENV2 K47: 1-based=K | 0-based=T  OK
DENV2 H149: 1-based=H | 0-based=A  OK
DENV2 V151: 1-based=V | 0-based=G  OK
DENV2 F279: 1-based=F | 0-based=T  OK
DENV3 K47: 1-based=K | 0-based=T  OK
DENV3 H149: 1-based=H | 0-based=Q  OK
DENV3 V151: 1-based=V | 0-based=G  OK
DENV3 F279: 1-based=F | 0-based=A  OK
DENV4 K47: 1-based=K | 0-based=T  OK
DENV4 H149: 1-based=H | 0-based=A  OK
DENV4 V151: 1-based=V | 0-based=G  OK
DENV4 F279: 1-based=F | 0-based=A  OK
DENV4 s[385:390] = ALTLH (esperado ALTLH, posiciones 38

## Calculo de matriz de distancias antigénicas

In [35]:
def _load_filtered_fasta(path: str, exclude_prefix: str) -> Tuple[List[str], List[str]]:
    """Lee un FASTA y devuelve (ids, secuencias), excluyendo IDs con cierto prefijo."""
    records = [
        r for r in SeqIO.parse(path, "fasta")
        if not r.id.startswith(exclude_prefix)
    ]
    return [r.id for r in records], [str(r.seq) for r in records]


def compute_distance_matrices(
    fasta_col_template: str,
    fasta_exp_template: str,
    exclude_prefix: str = "NODE",
    metric: str = "grantham",
) -> List[pd.DataFrame]:
    """Una matriz de distancias (colombianas x experimentales) por serotipo."""
    serotypes = [1, 2, 3, 4]

    # Referencias experimentales unificadas (todos los serotipos)
    ids_exp, seqs_exp = [], []
    for sero in serotypes:
        ids, seqs = _load_filtered_fasta(fasta_exp_template.format(sero), exclude_prefix)
        ids_exp.extend(ids)
        seqs_exp.extend(seqs)
    print(f"Conjunto unificado de referencias experimentales: {len(seqs_exp)} secuencias")

    matrices = []
    for sero in serotypes:
        print(f"Procesando DENV{sero}...")
        ids_col, seqs_col = _load_filtered_fasta(fasta_col_template.format(sero), exclude_prefix)

        matrix = np.full((len(seqs_col), len(seqs_exp)), np.nan)
        for i, seq_col in enumerate(seqs_col):
            for j, seq_exp in enumerate(seqs_exp):
                matrix[i, j] = antigenic_distance(seq_col, seq_exp, sero, type=metric)
        matrices.append(pd.DataFrame(matrix, index=ids_col, columns=ids_exp))

    return matrices


In [36]:
def _pad_denv3(seq: str) -> str:
    """DENV3 tiene dos residuos menos: deleción en 158-159 (numeración DENV1/2/4).
    Se insertan dos huecos para que los cuatro serotipos compartan numeración."""
    return seq[:157] + "--" + seq[157:]


def gene_distance(s1: str, s2: str, type: str = "grantham") -> float:
    """Distancia sobre todo el gen E, sin recortar a los sitios epitópicos.

    Se normaliza igual que la Ec. 2 (dividir por n y por el máximo de la métrica,
    escalar por 20) para que quede en la misma escala que la distancia epitópica
    y sirva como modelo nulo. Las columnas con hueco o stop se ignoran en ambas
    métricas, así que los dos huecos de DENV3 no cuentan como diferencias.
    """
    pairs = [(a, b) for a, b in zip(s1, s2)
             if a not in "-*" and b not in "-*"]
    if not pairs:
        return np.nan

    c1 = "".join(a for a, _ in pairs)
    c2 = "".join(b for _, b in pairs)
    total = grantham_distance(c1, c2) if type == "grantham" else hamming_distance(c1, c2)
    w = 215 if type == "grantham" else 1
    return (total / (len(pairs) * w)) * 20


def compute_gene_distance_matrices(
    fasta_col_template: str,
    fasta_exp_template: str,
    exclude_prefix: str = "NODE",
    metric: str = "grantham",
) -> List[pd.DataFrame]:
    """Como compute_distance_matrices, pero sobre todo el gen E."""
    serotypes = [1, 2, 3, 4]

    # Referencias experimentales unificadas (todos los serotipos)
    ids_exp, seqs_exp = [], []
    for sero in serotypes:
        ids, seqs = _load_filtered_fasta(fasta_exp_template.format(sero), exclude_prefix)
        ids_exp.extend(ids)
        seqs_exp.extend(_pad_denv3(s) if sero == 3 else s for s in seqs)
    print(f"Conjunto unificado de referencias experimentales: {len(seqs_exp)} secuencias")

    matrices = []
    for sero in serotypes:
        print(f"Procesando DENV{sero}...")
        ids_col, seqs_col = _load_filtered_fasta(fasta_col_template.format(sero), exclude_prefix)
        if sero == 3:
            seqs_col = [_pad_denv3(s) for s in seqs_col]

        matrix = np.full((len(seqs_col), len(seqs_exp)), np.nan)
        for i, seq_col in enumerate(seqs_col):
            for j, seq_exp in enumerate(seqs_exp):
                matrix[i, j] = gene_distance(seq_col, seq_exp, type=metric)
        matrices.append(pd.DataFrame(matrix, index=ids_col, columns=ids_exp))

    return matrices


### Matrices de distancia Grantaham epitópico

In [26]:
dfs_epi_grantham = compute_distance_matrices(AA_SEQS_COL_E, AA_SEQS_EXP_E_MAP)

Conjunto unificado de referencias experimentales: 385 secuencias
Procesando DENV1...
Procesando DENV2...
Procesando DENV3...
Procesando DENV4...


In [27]:
for i, sero in enumerate([1, 2, 3, 4]):
    filename = DIST_EPITOPE_GRANTHAM.format(sero=sero)
    dfs_epi_grantham[i].to_csv(filename, index=True, na_rep='NaN')
    
    print(f"Matriz de DENV{sero} exportada a: {filename}")

Matriz de DENV1 exportada a: ../output/distances_epitope_grantham_DENV1.csv
Matriz de DENV2 exportada a: ../output/distances_epitope_grantham_DENV2.csv
Matriz de DENV3 exportada a: ../output/distances_epitope_grantham_DENV3.csv
Matriz de DENV4 exportada a: ../output/distances_epitope_grantham_DENV4.csv


### Matrices validación de distancia Grantaham epitópico

In [28]:
dfs_epi_grantham_val = compute_distance_matrices(AA_SEQS_EXP_E_MAP, AA_SEQS_EXP_E_MAP)

Conjunto unificado de referencias experimentales: 385 secuencias
Procesando DENV1...
Procesando DENV2...
Procesando DENV3...
Procesando DENV4...


In [29]:
for i, sero in enumerate([1, 2, 3, 4]):
    filename = DIST_EPITOPE_GRANTHAM_VAL.format(sero=sero)
    dfs_epi_grantham_val[i].to_csv(filename, index=True, na_rep='NaN')
    
    print(f"Matriz de DENV{sero} exportada a: {filename}")

Matriz de DENV1 exportada a: ../output/validation_distances_epitope_grantham_DENV1.csv
Matriz de DENV2 exportada a: ../output/validation_distances_epitope_grantham_DENV2.csv
Matriz de DENV3 exportada a: ../output/validation_distances_epitope_grantham_DENV3.csv
Matriz de DENV4 exportada a: ../output/validation_distances_epitope_grantham_DENV4.csv


### Matrices validación de distancia todo gen E Grantaham

In [37]:
dfs_all_grantham_val = compute_gene_distance_matrices(
    AA_SEQS_EXP_E_MAP, AA_SEQS_EXP_E_MAP, metric="grantham")


Conjunto unificado de referencias experimentales: 385 secuencias
Procesando DENV1...
Procesando DENV2...
Procesando DENV3...
Procesando DENV4...


In [38]:
for i, sero in enumerate([1, 2, 3, 4]):
    filename = DIST_ALL_GRANTHAM_VAL.format(sero=sero)
    dfs_all_grantham_val[i].to_csv(filename, index=True, na_rep='NaN')

    print(f"Matriz de DENV{sero} exportada a: {filename}")


Matriz de DENV1 exportada a: ../output/validation_distances_all_grantham_DENV1.csv
Matriz de DENV2 exportada a: ../output/validation_distances_all_grantham_DENV2.csv
Matriz de DENV3 exportada a: ../output/validation_distances_all_grantham_DENV3.csv
Matriz de DENV4 exportada a: ../output/validation_distances_all_grantham_DENV4.csv


### Matrices validación de distancia Hamming epitópico

In [39]:
dfs_epi_hamming_val = compute_distance_matrices(
    AA_SEQS_EXP_E_MAP, AA_SEQS_EXP_E_MAP, metric="hamming")


Conjunto unificado de referencias experimentales: 385 secuencias
Procesando DENV1...
Procesando DENV2...
Procesando DENV3...
Procesando DENV4...


In [40]:
for i, sero in enumerate([1, 2, 3, 4]):
    filename = DIST_EPITOPE_HAMMING_VAL.format(sero=sero)
    dfs_epi_hamming_val[i].to_csv(filename, index=True, na_rep='NaN')

    print(f"Matriz de DENV{sero} exportada a: {filename}")


Matriz de DENV1 exportada a: ../output/validation_distances_epitope_hamming_DENV1.csv
Matriz de DENV2 exportada a: ../output/validation_distances_epitope_hamming_DENV2.csv
Matriz de DENV3 exportada a: ../output/validation_distances_epitope_hamming_DENV3.csv
Matriz de DENV4 exportada a: ../output/validation_distances_epitope_hamming_DENV4.csv


### Matrices validación de distancia todo gen E Hamming

In [41]:
dfs_all_hamming_val = compute_gene_distance_matrices(
    AA_SEQS_EXP_E_MAP, AA_SEQS_EXP_E_MAP, metric="hamming")


Conjunto unificado de referencias experimentales: 385 secuencias
Procesando DENV1...
Procesando DENV2...
Procesando DENV3...
Procesando DENV4...


In [42]:
for i, sero in enumerate([1, 2, 3, 4]):
    filename = DIST_ALL_HAMMING_VAL.format(sero=sero)
    dfs_all_hamming_val[i].to_csv(filename, index=True, na_rep='NaN')

    print(f"Matriz de DENV{sero} exportada a: {filename}")


Matriz de DENV1 exportada a: ../output/validation_distances_all_hamming_DENV1.csv
Matriz de DENV2 exportada a: ../output/validation_distances_all_hamming_DENV2.csv
Matriz de DENV3 exportada a: ../output/validation_distances_all_hamming_DENV3.csv
Matriz de DENV4 exportada a: ../output/validation_distances_all_hamming_DENV4.csv
